In [33]:
import time
from PIL import ImageGrab
import pyautogui
import numpy as np

# PyAutoGUI 安全機制：把滑鼠移到螢幕左上角 (0,0) 可緊急中斷
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0

# ==================== 設定參數 ====================
# 偵測區域
SCAN_LEFT = 60
SCAN_TOP = 336
SCAN_RIGHT = 135
SCAN_BOTTOM = 361

# 仙人掌目標顏色
COLOR_DAY = (83, 83, 83)       # 白天仙人掌顏色（深灰）
COLOR_NIGHT = (172, 172, 172)  # 夜間仙人掌顏色（淺灰）

# 顏色容差
TOLERANCE = 5

# 【基礎時間參數】（初始遊戲速度下的設定）
BASE_JUMP_WAIT = 0.28   # 初始起跳下壓等待時間
BASE_COOLDOWN = 0.001    # 初始跳躍冷卻時間
DROP_DURATION = 0.05    # 按住向下鍵時間

# 【最小時間下限】（防止速度過快時縮過短）
MIN_JUMP_WAIT = 0.10
MIN_COOLDOWN = 0.05
# ==================================================

def is_night_mode(arr):
    """透過檢查區域內的平均亮度來判斷當前是白天還是夜間模式"""
    avg_color = np.mean(arr)
    return avg_color < 120

def detect_obstacle(arr):
    """根據當前日夜模式，動態選擇要偵測的仙人掌顏色"""
    r = arr[:, :, 0]
    g = arr[:, :, 1]
    b = arr[:, :, 2]

    if is_night_mode(arr):
        tr, tg, tb = COLOR_NIGHT
    else:
        tr, tg, tb = COLOR_DAY

    mask = (
        (np.abs(r - tr) <= TOLERANCE) &
        (np.abs(g - tg) <= TOLERANCE) &
        (np.abs(b - tb) <= TOLERANCE)
    )

    count = np.count_nonzero(mask)
    return count > 3

def quick_jump(wait_time):
    """執行跳躍並依當前計算出的 wait_time 快速落地"""
    pyautogui.press('space')
    
    time.sleep(wait_time)
    pyautogui.keyDown('down')
    time.sleep(DROP_DURATION)
    pyautogui.keyUp('down')


def main():
    print("=" * 40)
    print("Chrome 小恐龍 - 自動日夜切換 + 動態加速版")
    print("=" * 40)
    print(f"偵測區域：({SCAN_LEFT}, {SCAN_TOP}) ~ ({SCAN_RIGHT}, {SCAN_BOTTOM})")
    print(f"初始起跳停留：{BASE_JUMP_WAIT} 秒（隨遊戲時間自動遞減加速）")
    print("\n[使用說明]")
    print("1. 開啟 chrome://dino/ 並按空白鍵開始遊戲")
    print("2. 腳本將於 5 秒後開始執行")
    print("3. 緊急停止：將滑鼠快速移至螢幕左上角 (0,0)")
    print("=" * 40)

    for i in range(5, 0, -1):
        print(f"倒數 {i} 秒後開始...")
        time.sleep(1)

    print("\n[開始執行] 不斷監視畫面中...")

    last_jump = 0
    was_detected = False
    game_start_time = None  # 紀錄當前一局的開始時間

    try:
        while True:
            now = time.time()

            # 根據遊戲進行時間，動態計算當前適用的等待時間與冷卻時間
            if game_start_time is not None:
                elapsed = now - game_start_time
                # 隨時間縮短等待時間（加速因子可依體驗微調）
                speed_factor = max(0.0, elapsed * 0.002)
                
                current_wait = max(MIN_JUMP_WAIT, BASE_JUMP_WAIT - speed_factor)
                current_cooldown = max(MIN_COOLDOWN, BASE_COOLDOWN - speed_factor)
            else:
                current_wait = BASE_JUMP_WAIT
                current_cooldown = BASE_COOLDOWN

            # 檢查跳躍冷卻
            if now - last_jump < current_cooldown:
                time.sleep(0.001)
                continue

            # 若超過 5 秒沒有觸發跳躍，自動重置遊戲計時（代表死掉重新開始）
            if game_start_time and (now - last_jump > 5.0):
                game_start_time = None

            # 讀取偵測區域
            img = ImageGrab.grab(bbox=(SCAN_LEFT, SCAN_TOP, SCAN_RIGHT, SCAN_BOTTOM))
            arr = np.array(img)

            # 動態判斷是否有仙人掌
            detected = detect_obstacle(arr)

            # 觸發跳躍與急降
            if detected and not was_detected:
                if game_start_time is None:
                    game_start_time = now  # 第一跳時記錄遊戲開局時間

                quick_jump(current_wait)
                last_jump = time.time()

            # 更新前一次狀態
            was_detected = detected

            time.sleep(0.001)

    except pyautogui.FailSafeException:
        print("\n[停止] 觸發 Failsafe（滑鼠已移至螢幕左上角）")
    except KeyboardInterrupt:
        print("\n[停止] 手動中斷")
    except Exception as e:
        print(f"\n[錯誤] {e}")


if __name__ == "__main__":
    main()

Chrome 小恐龍 - 自動日夜切換 + 動態加速版
偵測區域：(60, 336) ~ (135, 361)
初始起跳停留：0.28 秒（隨遊戲時間自動遞減加速）

[使用說明]
1. 開啟 chrome://dino/ 並按空白鍵開始遊戲
2. 腳本將於 5 秒後開始執行
3. 緊急停止：將滑鼠快速移至螢幕左上角 (0,0)
倒數 5 秒後開始...
倒數 4 秒後開始...
倒數 3 秒後開始...
倒數 2 秒後開始...
倒數 1 秒後開始...

[開始執行] 不斷監視畫面中...

[停止] 手動中斷
